<a href="https://colab.research.google.com/github/iaportnov/GP_2_real_estate_investment_strategy/blob/main/dataset_assemblance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import numpy as np
import json

from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

In [2]:
api_key = 'c2c86ae89cde0e20dadb510cf7a99a56'

headers = {
    'accept': 'application/json',
    'apikey': f'{api_key}'
}

Список наших признаков, которые мы будем извлекать из API:

In [3]:
features = [
    # community
    'geographyName',

    'median_Household_Income',
    'household_Income_Per_Capita',
    'housing_Median_Rent',
    'housing_Owner_Households_Median_Value',
    'housing_Units_Renter_Occupied_Pct',
    'housing_Units_Vacant_Pct',
    'median_Length_Of_Residence_Yr',
    'median_Age',
    'households_1_Person_Pct',
    'crime_Index',
    'annual_Avg_Temp',
    'possible_Sunshine_Pct',
    'air_Pollution_Index',

    'housing_Median_Built_Yr',
    'education_Bach_Degree_Pct',
    'education_Grad_Degree_Pct',
    'occupation_White_Collar_Pct',
    'occupation_Blue_Collar_Pct',
    'population_16P_Unemployed_Male_Pct',
    'population_16P_Unemployed_Female_Pct',
    'population_Chg_Pct_5_Yr_Projection',
    'population_Chg_Pct_2010',
    'population_White_Pct',
    'population_Black_Pct',
    'population_Hispanic_Pct',
    'population_Aged_25_34_Pct',
    'population_Aged_35_44_Pct',
    'population_Aged_65_74_Pct',

    'area_Square_Mile',
    'latitude',
    'longitude',
    'households_Family_W_Children_Pct',
    'population',
    'population_2020',
    'population_Male_Pct',
    'population_Female_Pct',
    'population_Aged_0_5_Pct',
    'population_Aged_6_11_Pct',
    'population_Aged_12_17_Pct',
    'population_Aged_18_24_Pct',
    'population_Aged_45_54_Pct',
    'population_Aged_55_64_Pct',
    'population_Aged_75_84_Pct',
    'population_Aged_85P_Pct',
    'costIndex_Annual_Expenditures',
    'costIndex_Housing',
    'costIndex_Healthcare',
    'costIndex_Education',
    'householder_Median_Age',
    'housing_Occupied_Structure_2_Units_Pct',
    'housing_Occupied_Structure_3_4_Units_Pct',
    'housing_Occupied_Structure_5_9_Units_Pct',
    'housing_Occupied_Structure_10_19_Units_Pct',
    'housing_Occupied_Structure_20_49_Units_Pct',
    'housing_Occupied_Structure_50_Or_More_Units_Pct',
    'housing_Built_2010_Or_Later_Pct',
    'housing_Built_2000_2009_Pct',
    'housing_Built_1990_1999_Pct',
    'housing_Built_1980_1989_Pct',
    'housing_Built_1970_1979_Pct',
    'housing_Built_1960_1969_Pct',
    'housing_Built_1950_1959_Pct',
    'housing_Built_1940_1949_Pct',
    'housing_Built_1939_Or_Earlier_Pct',
    'travel_Time_To_Work_0_14_Mi_Pct',
    'travel_Time_To_Work_15_29_Mi_Pct',
    'travel_Time_To_Work_30_59_Mi_Pct',
    'travel_Time_To_Work_60_89_Mi_Pct',
    'travel_Time_To_Work_90_Or_More_Mi_Pct',

    'homeSaleCount',
    'avgSalePrice',
    'medSalePrice',
    'pubDate'
]

Считываем раннее подготовленный датасет с выбранными районами:

In [4]:
all_neighborhoods = pd.read_csv('all_neighborhoods.csv')

list_of_dicts = all_neighborhoods.to_dict('records')
# list_of_dicts

Функция сбора данных, подчиненных полю community:

In [5]:
def get_community_data(geo_id):
  url = 'https://api.gateway.attomdata.com/v4/neighborhood/community'
  params = {'geoIdv4': geo_id}

  request = requests.get(url, headers=headers, params=params)
  data = request.json()

  return data['community']

Функция извлечения наших выбранных признаков:

In [6]:
def extract_features(community_data):
  geography = community_data['geography']
  demographics = community_data['demographics']
  crime = community_data['crime']
  climate = community_data['climate']
  air_quality = community_data['airQuality']

  result = {}
  for feature in features:
    if feature in geography:
      result[feature] = geography.get(feature)
    elif feature in demographics:
      result[feature] = demographics.get(feature)
    elif feature in crime:
      result[feature] = crime.get(feature)
    elif feature in climate:
      result[feature] = climate.get(feature)
    elif feature in air_quality:
      result[feature] = air_quality.get(feature)
    else:
      result[feature] = None
  return result

Функция для сплита колонки geographyName:

In [7]:
def get_geography_names(geography_name):
  all_names = geography_name.split(',')
  if len(all_names) == 3:
    return [None, all_names[0], all_names[1], all_names[2]]
  return all_names

Функция подсчета кол-ва медицинских, развлекательных, образовательных и других учреждений:

In [8]:
def get_poi_counts(lat, lon, radius):
  url = 'https://api.gateway.attomdata.com/v4/neighborhood/poi'

  params = {
      'point': f'POINT({lon},{lat})',
      'radius': radius,
      'page': 1,
      'pagesize': 500
  }

  categories = {
      'healthcare': {'categories': set(), 'line_of_businesses': {'PHYSICIANS AND SURGEONS', 'PHARMACIES', 'DENTISTS', 'HOSPITALS', 'URGENT CARE', 'MEDICAL CLINICS', 'NURSING AND PERSONAL CARE FACILITIES', 'CHILDREN HEALTH CARE'}},
      'grocery': {'categories': set(), 'line_of_businesses': {'GROCERY STORES AND MARKETS'}},
      'education': {'categories': {'EDUCATION'}, 'line_of_businesses': {'PUBLIC SCHOOL', 'PRIVATE SCHOOL', 'CATHOLIC SCHOOL', 'VOCATIONAL', 'CHILD CARE'}},
      'entertainment': {'categories': set(), 'line_of_businesses': {'ART AND MUSEUMS', 'MOVIE THEATERS', 'AMUSEMENT PLACES', 'MISC ATTRACTIONS', 'BARS - CLUBS'}},
      'recreation': {'categories': {'HOSPITALITY'}, 'line_of_businesses': {'CAMPS AND CAMPGROUNDS', 'HEALTH CLUBS AND SPAS'}}
  }

  counts = {k: 0 for k in categories}
  request = requests.get(url, headers=headers, params=params)
  request.raise_for_status()
  data = request.json()

  poi_data = data.get('poi', [])

  for poi in poi_data:
    category = poi['category']['category'].upper()
    line_of_business = poi['category']['lineOfBusiness'].upper()

    for k, cat in categories.items():
      if (category in cat['categories'] or line_of_business in cat['line_of_businesses']):
        counts[k] += 1
        break

  return counts

Функция финальной сборки:

In [9]:
def extract_data(row):
  community_data = get_community_data(row['geo_id'])
  features_data = extract_features(community_data)
  poi_counts = get_poi_counts(row['latitude'], row['longitude'], radius=5) # радиус здесь задаем
  result = row.to_dict()
  result.update(features_data)
  result.update(poi_counts)

  return result

In [11]:
rows = [row for _, row in all_neighborhoods.iterrows()]

Сборка итогового датасета c использованием Thread Pool Executor (многопоточности):

In [12]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

with ThreadPoolExecutor(max_workers=20) as executor:
    final_data = list(tqdm(executor.map(extract_data, rows[:50]), total=len(rows[:50])))

df = pd.DataFrame(final_data)

splited = df['geographyName'].str.split(', ', expand=True)
df[['neighborhood', 'city', 'county', 'state']] = df['geographyName'].apply(lambda x: pd.Series(get_geography_names(x)))

100%|██████████| 50/50 [00:09<00:00,  5.35it/s]


Переименуем кое-какие колонки:

In [13]:
df.rename(columns={
    'healthcare': 'healthcare_count',
    'grocery': 'grocery_count',
    'education': 'education_count',
    'entertainment': 'entertainment_count',
    'recreation': 'recreation_count'
}, inplace=True)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 85 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   geo_id                                           50 non-null     object 
 1   latitude                                         50 non-null     float64
 2   longitude                                        50 non-null     float64
 3   area                                             50 non-null     float64
 4   geographyName                                    50 non-null     object 
 5   median_Household_Income                          50 non-null     int64  
 6   household_Income_Per_Capita                      50 non-null     int64  
 7   housing_Median_Rent                              50 non-null     int64  
 8   housing_Owner_Households_Median_Value            50 non-null     int64  
 9   housing_Units_Renter_Occupied_Pct 

In [40]:
df

,geo_id,latitude,longitude,area,geographyName,median_Household_Income,household_Income_Per_Capita,housing_Median_Rent,housing_Owner_Households_Median_Value,housing_Units_Renter_Occupied_Pct,...,travel_Time_To_Work_90_Or_More_Mi_Pct,healthcare,grocery,education,entertainment,recreation,neighborhood,city,county,state
0,c4985358751ed9e97f106aa9dc0aa640,39.753611,-105.112466,1.549311,"Acipco / Finley, Birmingham, Jefferson County, AL",33267,16911,475,60879,55.47,...,2.36,4,27,11,1,1,Acipco / Finley,Birmingham,Jefferson County,AL
1,7b8a4ad71938b16e13d0bc8e19fee22a,30.727964,-88.058559,0.828192,"Africatown, Mobile, Mobile County, AL",46193,11511,363,64402,50.70,...,0.00,1,27,42,5,3,Africatown,Mobile,Mobile County,AL
2,5df42aba382587eeadad52a615e8cef7,28.040854,-97.028924,1.587457,"Airmont, Mobile, Mobile County, AL",47083,24760,532,162720,53.82,...,3.44,2,1,9,6,6,Airmont,Mobile,Mobile County,AL
3,b060e1f941f76c9b6e6887a7db8ee21b,41.984866,-87.671462,0.270328,"Airport Highlands, Birmingham, Jefferson Count...",56903,21178,746,117295,59.93,...,0.00,3,10,24,3,5,Airport Highlands,Birmingham,Jefferson County,AL
4,7db649e182afe8c6c6595af9348909d7,29.290475,-81.084510,1.568713,"Alderbrook, Mobile County, AL",95479,33127,1136,291547,17.91,...,7.39,6,12,14,7,6,None,Alderbrook,Mobile County,AL
5,29e05661182f45955b543f34fb6dd927,42.406419,-71.120848,1.969267,"Alligator Bayou, Mobile County, AL",61703,39474,861,438919,7.89,...,0.05,0,4,15,11,2,None,Alligator Bayou,Mobile County,AL
6,5aa3ceb3ce301822581097a88839779d,41.812704,-87.617760,0.525244,"Apple Valley, Birmingham, Jefferson County, AL",34191,21875,413,133867,61.33,...,0.26,2,12,25,4,6,Apple Valley,Birmingham,Jefferson County,AL
7,141842d75c20ca848eff00dc358d9c76,30.508298,-88.146751,2.698374,"Argyle, Mobile County, AL",72500,43080,575,126984,16.63,...,0.00,1,7,6,8,9,None,Argyle,Mobile County,AL
8,9a8984b2d0eab6d7d2fde79bbc09c5b5,37.209302,-113.267231,0.349425,"Arlington, Mobile, Mobile County, AL",22877,11443,229,102763,83.52,...,0.02,2,21,21,6,1,Arlington,Mobile,Mobile County,AL
9,c70fe7e35bbf0efdee6fc909792404cf,39.496198,-104.774152,1.754165,"Arlington / West End, Birmingham, Jefferson Co...",26019,19323,555,67680,60.31,...,5.76,22,17,20,4,1,Arlington / West End,Birmingham,Jefferson County,AL
